# 02: Data Cleaning & Pipeline
This notebook implements the data cleaning pipeline to convert raw healthcare data into a standardized format for analysis.

## Import Requirements
Setting up the environment for data manipulation.

In [ ]:
import pandas as pd
import numpy as np

## Load Dataset
Retrieving the raw CSV file.

In [ ]:
df = pd.read_csv('../data/raw/diabetic_data.csv')

## Step 1: Standardize Missing Values
The dataset uses `?` to represent missing data. These are converted to standard NaN values.

In [ ]:
df.replace('?', np.nan, inplace=True)
df.isnull().sum().sort_values(ascending=False).head(10)

## Step 2: Handle Columns with High Missingness
Columns with excessive missing values (>40%) are dropped to maintain data quality.

In [ ]:
cols_to_drop = ['weight', 'payer_code', 'medical_specialty']
df.drop(columns=cols_to_drop, inplace=True)

## Step 3: Standardize Age Bins
Converting categorical age intervals into numerical midpoints for correlation analysis.

In [ ]:
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, 
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, 
    '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map)

## Step 4: Create Binary Target Variable
Targeting the 30-day readmission KPI by encoding `<30` as 1 and all other outcomes as 0.

In [ ]:
df['readmit_30d'] = (df['readmitted'] == '<30').astype(int)
df.drop(columns=['readmitted'], inplace=True)

## Step 5: Clean Demographic Inconsistencies
Filtering out invalid gender records and handling remaining missing values in 'race'.

In [ ]:
df = df[df['gender'] != 'Unknown/Invalid']
df['race'].fillna('Other', inplace=True)

## Step 6: Export Processed Data
Saving the final cleaned dataset for downstream analysis.

In [ ]:
output_path = '../data/processed/cleaned_diabetic_data.csv'
df.to_csv(output_path, index=False)